Exercise 1: Classes Store Type Hints

In [1]:
class TradeSchema:
    symbol:str
    price:float
    volume:int

print(TradeSchema.__annotations__)

{'symbol': <class 'str'>, 'price': <class 'float'>, 'volume': <class 'int'>}


Exercise 2: Extract Schema Fields

In [2]:
class TradeSchema:
    symbol: str
    price: float
    volume: int
    _internal_cache: dict

annotations = TradeSchema.__annotations__
print(f"Annotations: {annotations}")

schema_fields = {
    name: field_type
    for name, field_type in annotations.items()
    if not name.startswith("_")
}

print(schema_fields)

Annotations: {'symbol': <class 'str'>, 'price': <class 'float'>, 'volume': <class 'int'>, '_internal_cache': <class 'dict'>}
{'symbol': <class 'str'>, 'price': <class 'float'>, 'volume': <class 'int'>}


Purpose: learn how your framework separates real schema fields from internal fields.

Exercise 3: Simple Data Validation

In [3]:
class TradeSchema:
    symbol: str
    price: float
    volume: int

schema_fields = TradeSchema.__annotations__
print(f"Schema fields: {schema_fields}")


data = {
    "symbol": "AAPL",
    "price": 210.5,
    "volume": 1000
}

for field_name, expected_type in schema_fields.items():
    value = data.get(field_name)

    if not isinstance(value, expected_type):
        print(f"{field_name} is invalid")
    else:
        print(f"{field_name} is valid")

Schema fields: {'symbol': <class 'str'>, 'price': <class 'float'>, 'volume': <class 'int'>}
symbol is valid
price is valid
volume is valid


In [4]:
bad_data = {
    "symbol": "AAPL",
    "price": "210.5",
    "volume": "1000"
}

for field_name, expected_type in schema_fields.items():
    value = bad_data.get(field_name)

    if not isinstance(value, expected_type):
        print(f"{field_name} is invalid")
    else:
        print(f"{field_name} is valid")

symbol is valid
price is invalid
volume is invalid


Exercise 4: Add a validate() Function Manually

In [5]:
def validate(schema_fields, data):
    errors = []

    for field_name, expected_type in schema_fields.items():
        value = data.get(field_name)

        if value is None:
            errors.append(f"{field_name} is missing")
        elif not isinstance(value, expected_type):
            errors.append(
                f"{field_name} should be {expected_type.__name__}, got {type(value).__name__}"
            )

    return errors

class TradeSchema:
    symbol: str
    price: float
    volume: int


errors = validate(
    TradeSchema.__annotations__,
    {
        "symbol": "AAPL",
        "price": "210.5",
        "volume": 1000
    }
)

print(errors)

['price should be float, got str']


Exercise 5: Attach validate() to a Class Manually

In [6]:
class TradeSchema:
    symbol: str
    price: float
    volume: int


def validate_trade(data):
    return validate(TradeSchema.__annotations__, data)


TradeSchema.validate = staticmethod(validate_trade)

result = TradeSchema.validate({
    "symbol": "AAPL",
    "price": 210.5,
    "volume": 1000
})

print(result)

[]


Exercise 6: Build a Simple Registry

In [7]:
SCHEMA_REGISTRY = {}


class TradeSchema:
    symbol: str
    price: float
    volume: int


SCHEMA_REGISTRY["TradeSchema"] = TradeSchema.__annotations__

print(SCHEMA_REGISTRY)

{'TradeSchema': {'symbol': <class 'str'>, 'price': <class 'float'>, 'volume': <class 'int'>}}


Exercise 7: Register a Class Using a Decorator

In [8]:
SCHEMA_REGISTRY = {}


def contract(cls):
    SCHEMA_REGISTRY[cls.__name__] = cls.__annotations__
    return cls


@contract
class TradeSchema:
    symbol: str
    price: float
    volume: int


print(SCHEMA_REGISTRY)

{'TradeSchema': {'symbol': <class 'str'>, 'price': <class 'float'>, 'volume': <class 'int'>}}


This is the first big “aha” moment.

The decorator automatically registers the schema.

Exercise 8: Decorator Adds validate()

In [ ]:
SCHEMA_REGISTRY = {}


def contract(cls):
    # gets the schema fields from the class annotations
    schema_fields = cls.__annotations__
    
    # saves the schema fields in the registry
    SCHEMA_REGISTRY[cls.__name__] = schema_fields

    # defines a validate method that uses the schema fields to validate the data
    def _validate(data):
        return validate(schema_fields, data)

    cls.validate = staticmethod(_validate)

    return cls


@contract
class TradeSchema:
    symbol: str
    price: float
    volume: int


print(SCHEMA_REGISTRY)

print(
    TradeSchema.validate({
        "symbol": "AAPL",
        "price": "210.5",
        "volume": 1000
    })
)

{'TradeSchema': {'symbol': <class 'str'>, 'price': <class 'float'>, 'volume': <class 'int'>}}
['price should be float, got str']


In [12]:
print(TradeSchema.__name__)
print(TradeSchema.__annotations__)

TradeSchema
{'symbol': <class 'str'>, 'price': <class 'float'>, 'volume': <class 'int'>}


Exercise 9: Add Versioning

In [13]:
SCHEMA_REGISTRY = {}


def contract(version="1.0.0"):

    def decorator(cls):
        schema_fields = cls.__annotations__

        SCHEMA_REGISTRY[cls.__name__] = {
            "version": version,
            "fields": schema_fields
        }

        return cls

    return decorator


@contract(version="1.0.0")
class TradeSchema:
    symbol: str
    price: float
    volume: int


print(SCHEMA_REGISTRY)

{'TradeSchema': {'version': '1.0.0', 'fields': {'symbol': <class 'str'>, 'price': <class 'float'>, 'volume': <class 'int'>}}}


Exercise 10: First Metaclass

In [18]:
class SimpleMeta(type):

    def __new__(mcs, name, bases, namespace):
        print(f"Creating class: {name}")
        print(f"Namespace: {namespace}")
        print(f"bases: {bases}")
        print(f"minetaclass: {mcs}")

        return super().__new__(mcs, name, bases, namespace)


class TradeSchema(metaclass=SimpleMeta):
    symbol: str
    price: float
    volume: int

Creating class: TradeSchema
Namespace: {'__module__': '__main__', '__qualname__': 'TradeSchema', '__annotations__': {'symbol': <class 'str'>, 'price': <class 'float'>, 'volume': <class 'int'>}}
bases: ()
minetaclass: <class '__main__.SimpleMeta'>


Exercise 11: Metaclass Registers Schema Automatically

In [16]:
SCHEMA_REGISTRY = {}


class ContractMeta(type):

    def __new__(mcs, name, bases, namespace):
        cls = super().__new__(mcs, name, bases, namespace)

        schema_fields = namespace.get("__annotations__", {})

        SCHEMA_REGISTRY[name] = schema_fields

        return cls


class TradeSchema(metaclass=ContractMeta):
    symbol: str
    price: float
    volume: int


print(SCHEMA_REGISTRY)

{'TradeSchema': {'symbol': <class 'str'>, 'price': <class 'float'>, 'volume': <class 'int'>}}


Exercise 12: Use a Base Class

In [17]:
SCHEMA_REGISTRY = {}


class ContractMeta(type):

    def __new__(mcs, name, bases, namespace):
        cls = super().__new__(mcs, name, bases, namespace)

        if name != "ContractBase":
            schema_fields = namespace.get("__annotations__", {})
            SCHEMA_REGISTRY[name] = schema_fields

        return cls


class ContractBase(metaclass=ContractMeta):
    pass


class TradeSchema(ContractBase):
    symbol: str
    price: float
    volume: int


print(SCHEMA_REGISTRY)

{'TradeSchema': {'symbol': <class 'str'>, 'price': <class 'float'>, 'volume': <class 'int'>}}


This is closer to your actual framework.

Now any class that inherits from ContractBase automatically becomes a registered schema.

The reason we use a base class here in the middle is, if we want to use the meta class in our class generation we need to say metaclass=ContractMeta everytime.

Someone will forget. So we create a base class and any schema just have to inherit it like 

class TradeSchema(ContractBase):
    ...

Which essentially saves time


First: What is __mro__?

MRO stands for:

Method Resolution Order

It tells Python:

"If I can't find something in this class, where should I look next?"

Consider:

In [19]:
class Animal:
    pass

class Dog(Animal):
    pass

print(Dog.__mro__)

(<class '__main__.Dog'>, <class '__main__.Animal'>, <class 'object'>)


Why Does Python Need This?

In [21]:
class Animal:
    sound = "generic"

class Dog(Animal):
    sound = "bark"

print(Dog.sound)

bark


It finds sound in Dog first and stops.

In [23]:
class BaseSchema:
    created_at: str

class TradeSchema(BaseSchema):
    symbol: str
    price: float

print(TradeSchema.__annotations__)

{'symbol': <class 'str'>, 'price': <class 'float'>}


We dont get created_at from the parent class.

Python stores annotations separately for each class.

But we need created_at as well.

So this code

```
annotations: dict[str, Any] = {}
for base in reversed(cls.__mro__):
    annotations.update(getattr(base, "__annotations__", {}))

```
Solves that issue by taking each class starting from the base and taking the annotations separately


In [25]:
for base in reversed(TradeSchema.__mro__):
    print(base.__name__)


object
BaseSchema
TradeSchema


In [33]:
annotations = {}
for base in reversed(TradeSchema.__mro__):
    print(base.__name__)
    annotations.update(getattr(base, "__annotations__", {}))
print(annotations)


object
BaseSchema
TradeSchema
{'created_at': <class 'str'>, 'price': <class 'float'>, 'symbol': <class 'str'>}


In [35]:
from typing import get_type_hints
resolved = get_type_hints(TradeSchema)
print(resolved)

{'created_at': <class 'str'>, 'price': <class 'float'>, 'symbol': <class 'str'>}


We reverse it because it allows us to avoid overwriting existing fields

In [27]:
class BaseSchema:
    created_at: str
    price: int


class TradeSchema(BaseSchema):
    symbol: str
    price: float


annotations = {}

for base in reversed(TradeSchema.__mro__):
    print(f"Looking at {base.__name__}")

    annotations.update(
        getattr(base, "__annotations__", {})
    )

    print(annotations)
    print()

Looking at object
{}

Looking at BaseSchema
{'created_at': <class 'str'>, 'price': <class 'int'>}

Looking at TradeSchema
{'created_at': <class 'str'>, 'price': <class 'float'>, 'symbol': <class 'str'>}

